## **Part 1: How strong is this El Niño?**
This notebook identifies past El Niño and La Niña events from NOAA’s Oceanic Niño Index (ONI) and compares them with the 2026 event. ONI is the three-month average sea surface temperature anomaly in the central Pacific (the Niño 3.4 region), in °C.

### Data and Labels

#### 1. Import libraries

In [1]:
import pandas as pd
import numpy as np

#### 2. Load ONI from NOAA into a DataFrame
The data is read directly from NOAA’s Climate Prediction Center. The ANOM column holds the ONI values; TOTAL is the actual sea surface temperature.

In [2]:
url = 'https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt'
df = pd.read_csv(url, sep=r"\s+")
print(df.head())

  SEAS    YR  TOTAL  ANOM
0  DJF  1950  25.01 -1.32
1  JFM  1950  25.36 -1.20
2  FMA  1950  25.88 -1.12
3  MAM  1950  26.24 -1.08
4  AMJ  1950  26.35 -1.10


#### 3. Convert seasons to real dates (`datetime64`), with a check that no dates are missing
Each ONI value is a three-month average, labeled by season (for example, DJF = December–January–February). Each season is assigned to its middle month, so DJF becomes January. Any season that fails to convert would show up as a missing date, so the count below should be 0.

In [3]:
df["DATE"] = df["SEAS"].replace(["DJF", "JFM", "FMA", "MAM", "AMJ", "MJJ", "JJA", "JAS", "ASO", "SON", "OND", "NDJ"], ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"])
df["DATE"] = df["DATE"] + " " + df["YR"].astype(str)
df["DATE"] = pd.to_datetime(df["DATE"], format="%B %Y", errors="coerce")
print(f"Number of dates missing: {df['DATE'].isnull().sum()} \n")
print(df.head())

Number of dates missing: 0 

  SEAS    YR  TOTAL  ANOM       DATE
0  DJF  1950  25.01 -1.32 1950-01-01
1  JFM  1950  25.36 -1.20 1950-02-01
2  FMA  1950  25.88 -1.12 1950-03-01
3  MAM  1950  26.24 -1.08 1950-04-01
4  AMJ  1950  26.35 -1.10 1950-05-01


#### 4. Round ONI to one decimal (NOAA precision) and label each month El Niño / La Niña / Neutral
NOAA’s official ONI table rounds values to one decimal and labels months from the rounded values, so this notebook does the same (see [decision 2](https://github.com/zphrc/elnino-visayas-forecast/blob/main/docs/decisions.md#2-round-oni-to-one-decimal-before-labeling)). A month is El Niño if ONI is at or above +0.5, La Niña if at or below −0.5, and Neutral otherwise.

In [4]:
df["ONI"] = df["ANOM"]
df["ONI"] = df["ONI"].round(1)
df["LABEL"] = np.where(df["ONI"] >= 0.5, "El Niño", np.where(df["ONI"] <= -0.5, "La Niña", "Neutral"))
df_reordered = df[["DATE", "SEAS", "YR", "TOTAL", "ONI", "LABEL"]].copy()
print(df_reordered.head())

        DATE SEAS    YR  TOTAL  ONI    LABEL
0 1950-01-01  DJF  1950  25.01 -1.3  La Niña
1 1950-02-01  JFM  1950  25.36 -1.2  La Niña
2 1950-03-01  FMA  1950  25.88 -1.1  La Niña
3 1950-04-01  MAM  1950  26.24 -1.1  La Niña
4 1950-05-01  AMJ  1950  26.35 -1.1  La Niña


#### 5. Number consecutive runs of the same label
A run is a stretch of consecutive months with the same label. A new run starts whenever the label changes from the previous month, so a cumulative sum of those changes gives every month its run number.

In [5]:
df_reordered["RUN"] = (df_reordered["LABEL"] != df_reordered["LABEL"].shift()).cumsum()
print(df_reordered.head(50))

         DATE SEAS    YR  TOTAL  ONI    LABEL  RUN
0  1950-01-01  DJF  1950  25.01 -1.3  La Niña    1
1  1950-02-01  JFM  1950  25.36 -1.2  La Niña    1
2  1950-03-01  FMA  1950  25.88 -1.1  La Niña    1
3  1950-04-01  MAM  1950  26.24 -1.1  La Niña    1
4  1950-05-01  AMJ  1950  26.35 -1.1  La Niña    1
5  1950-06-01  MJJ  1950  26.32 -0.9  La Niña    1
6  1950-07-01  JJA  1950  26.24 -0.6  La Niña    1
7  1950-08-01  JAS  1950  25.94 -0.6  La Niña    1
8  1950-09-01  ASO  1950  25.76 -0.6  La Niña    1
9  1950-10-01  SON  1950  25.60 -0.6  La Niña    1
10 1950-11-01  OND  1950  25.55 -0.7  La Niña    1
11 1950-12-01  NDJ  1950  25.41 -0.8  La Niña    1
12 1951-01-01  DJF  1951  25.67 -0.7  La Niña    1
13 1951-02-01  JFM  1951  26.20 -0.4  Neutral    2
14 1951-03-01  FMA  1951  26.96 -0.0  Neutral    2
15 1951-04-01  MAM  1951  27.48  0.2  Neutral    2
16 1951-05-01  AMJ  1951  27.72  0.3  Neutral    2
17 1951-06-01  MJJ  1951  27.72  0.5  El Niño    3
18 1951-07-01  JJA  1951  27.57

In [6]:
df_reordered["RUN"].max() # number of runs

np.int64(109)

### Events

#### 1. Summarize runs and add **PEAK_ONI** to each run
Each run becomes one row, with its start, end, length, and label. The peak is the highest ONI for El Niño runs and the lowest for La Niña runs. Neutral runs have no peak, so they’re left empty.

In [7]:
runs = df_reordered.groupby("RUN").agg(
    START=("DATE", "min"),
    END=("DATE", "max"),
    MONTHS=("DATE", "count"),
    LABEL=("LABEL", "first"),
    PEAK_ONI=("ONI", "max"),
    BASE_ONI=("ONI", "min")
)

runs["PEAK_ONI"] = np.where(runs["LABEL"] == "El Niño", runs["PEAK_ONI"], np.where(runs["LABEL"] == "La Niña", runs["BASE_ONI"], np.nan))
runs = runs[["START", "END", "MONTHS", "LABEL", "PEAK_ONI"]]

print(runs.tail(50))

         START        END  MONTHS    LABEL  PEAK_ONI
RUN                                                 
60  1993-07-01 1994-08-01      14  Neutral       NaN
61  1994-09-01 1995-03-01       7  El Niño       1.0
62  1995-04-01 1995-07-01       4  Neutral       NaN
63  1995-08-01 1996-03-01       8  La Niña      -1.0
64  1996-04-01 1997-04-01      13  Neutral       NaN
65  1997-05-01 1998-04-01      12  El Niño       2.4
66  1998-05-01 1998-06-01       2  Neutral       NaN
67  1998-07-01 2000-07-01      25  La Niña      -1.5
68  2000-08-01 2000-09-01       2  Neutral       NaN
69  2000-10-01 2001-02-01       5  La Niña      -0.8
70  2001-03-01 2002-06-01      16  Neutral       NaN
71  2002-07-01 2003-02-01       8  El Niño       1.2
72  2003-03-01 2004-07-01      17  Neutral       NaN
73  2004-08-01 2005-02-01       7  El Niño       0.7
74  2005-03-01 2005-10-01       8  Neutral       NaN
75  2005-11-01 2006-03-01       5  La Niña      -0.9
76  2006-04-01 2006-08-01       5  Neutral    

#### 2. Filter to real events
Following NOAA’s definition, an El Niño or La Niña only counts as an event if it lasts at least five consecutive overlapping seasons.

*Check: the 1997–98, 2015–16, and 2023–24 El Niños all appear, with peaks of 2.4, 2.6, and 2.0.*

In [8]:
events = runs[((runs["LABEL"] == "El Niño") | (runs["LABEL"] == "La Niña")) & (runs["MONTHS"] >= 5)].copy()
print(events.head(100))

         START        END  MONTHS    LABEL  PEAK_ONI
RUN                                                 
1   1950-01-01 1951-01-01      13  La Niña      -1.3
3   1951-06-01 1952-01-01       8  El Niño       1.0
5   1953-02-01 1953-12-01      11  El Niño       0.8
7   1954-05-01 1954-10-01       6  La Niña      -0.8
9   1955-01-01 1956-08-01      20  La Niña      -1.6
11  1957-04-01 1958-06-01      15  El Niño       1.7
15  1963-07-01 1964-02-01       8  El Niño       1.2
17  1964-05-01 1965-01-01       9  La Niña      -0.8
19  1965-05-01 1966-03-01      11  El Niño       1.8
25  1968-10-01 1969-06-01       9  El Niño       1.0
27  1969-09-01 1970-01-01       5  El Niño       0.8
29  1970-07-01 1972-01-01      19  La Niña      -1.4
31  1972-05-01 1973-03-01      11  El Niño       1.8
33  1973-05-01 1974-07-01      15  La Niña      -2.0
35  1974-09-01 1976-03-01      19  La Niña      -1.6
37  1976-09-01 1977-02-01       6  El Niño       0.8
41  1977-10-01 1978-02-01       5  El Niño    

#### 3. Add a **STRENGTH** column
Strength is based on the size of the peak, regardless of sign: weak (0.5–0.9), moderate (1.0–1.4), strong (1.5–1.9), and very strong (2.0 and above).

In [9]:
condlist = [(events["PEAK_ONI"].abs() >= 0.5) & (events["PEAK_ONI"].abs() <= 0.9), (events["PEAK_ONI"].abs() >= 1.0) & (events["PEAK_ONI"].abs() <= 1.4), (events["PEAK_ONI"].abs() >= 1.5) & (events["PEAK_ONI"].abs() <= 1.9), events["PEAK_ONI"].abs() >= 2.0]
choicelist = ["weak", "moderate", "strong", "very strong"]
events["STRENGTH"] = np.select(condlist, choicelist, "unknown")
events = events[["START", "END", "MONTHS", "LABEL", "PEAK_ONI", "STRENGTH"]]
print(events.head(100))

         START        END  MONTHS    LABEL  PEAK_ONI     STRENGTH
RUN                                                              
1   1950-01-01 1951-01-01      13  La Niña      -1.3     moderate
3   1951-06-01 1952-01-01       8  El Niño       1.0     moderate
5   1953-02-01 1953-12-01      11  El Niño       0.8         weak
7   1954-05-01 1954-10-01       6  La Niña      -0.8         weak
9   1955-01-01 1956-08-01      20  La Niña      -1.6       strong
11  1957-04-01 1958-06-01      15  El Niño       1.7       strong
15  1963-07-01 1964-02-01       8  El Niño       1.2     moderate
17  1964-05-01 1965-01-01       9  La Niña      -0.8         weak
19  1965-05-01 1966-03-01      11  El Niño       1.8       strong
25  1968-10-01 1969-06-01       9  El Niño       1.0     moderate
27  1969-09-01 1970-01-01       5  El Niño       0.8         weak
29  1970-07-01 1972-01-01      19  La Niña      -1.4     moderate
31  1972-05-01 1973-03-01      11  El Niño       1.8       strong
33  1973-0

#### 4. Find the **2026 event**
The most recent event is the one with the latest end date. If that end date matches the last month in the data, the event hasn’t ended yet, so it’s ongoing.

In [10]:
filtered_events = events[events["END"] == events["END"].max()]
print(filtered_events)

print(f"\nLatest event is ongoing: {filtered_events['END'].max() == df_reordered['DATE'].max()}")


         START        END  MONTHS    LABEL  PEAK_ONI     STRENGTH
RUN                                                              
109 2026-04-01 2026-09-01       6  El Niño       2.5  very strong

Latest event is ongoing: True


#### **Observation**

According to the Oceanic Niño Index (ONI), as of the August-October season, the ongoing 2026 El Niño started in April and has lasted for 6 months with a peak ONI of 2.5 so far.

NOAA notes that the most recent values are preliminary and may be revised for up to two months.